# Clinical Notes Registry Builder: LLM extraction

This notebook reads de-identified orthopedic operative notes (`notes.csv`, built from the public
MTSamples dataset) and asks **Gemma 3 4B (instruction-tuned)** to extract four fields per note,
each with an exact evidence quote copied from the note:

| Field | Values |
|---|---|
| `procedure` | one or more of 9 fixed procedure classes |
| `body_part` | short anatomy term, e.g. `knee`, `lumbar spine` |
| `laterality` | `left` / `right` / `bilateral` / `not mentioned` |
| `patient_age` | integer, or `not mentioned` |

Output is `results.json`, which the Django backend imports. Flagging (quote not found, value outside the
allowed options, etc.) happens in the backend, not here. This notebook only checks that the reply is
well-formed JSON with the expected shape.

**Demo only, not for clinical use.**

### Before you run
1. **Runtime → Change runtime type → T4 GPU** (free tier is fine).
2. Accept the Gemma licence at https://huggingface.co/google/gemma-3-4b-it (logged in to Hugging Face).
3. Create a Hugging Face access token (read scope) and add it as a Colab secret named **`HF_TOKEN`**
   (🔑 icon in the left sidebar), with notebook access switched on.
4. Upload `data/notes.csv` from the repo via the 📁 Files panel so it sits at `/content/notes.csv`.

## 1. Install packages

In [ ]:
# Colab already has torch and pandas. Gemma 3 needs a recent transformers; pin it so runs are reproducible.
%pip install -q "transformers==4.57.1" accelerate

## 2. Settings

In [ ]:
import os

MODEL_ID = "google/gemma-3-4b-it"
NOTES_PATH = "notes.csv"
RESULTS_PATH = "results.json"
TEST_RESULTS_PATH = "results_test.json"
SAVE_EVERY = 10          # checkpoint to disk after this many notes
MAX_NEW_TOKENS = 768     # enough for the JSON reply; long notes have several procedures + quotes
PROMPT_VERSION = "v3"    # bump if you change the prompt, so results record which prompt made them

# Optional: save results to Google Drive so they survive a runtime reset
# (files in /content are deleted when Colab recycles the machine).
USE_DRIVE = False
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    out_dir = "/content/drive/MyDrive/clinical-registry-builder"
    os.makedirs(out_dir, exist_ok=True)
    RESULTS_PATH = os.path.join(out_dir, "results.json")
    TEST_RESULTS_PATH = os.path.join(out_dir, "results_test.json")

print("Results will be written to", RESULTS_PATH)

## 3. Log in to Hugging Face (token from Colab secrets)

In [ ]:
from google.colab import userdata
from huggingface_hub import login

# The token is read from Colab's secret store. Never paste it into the notebook.
try:
    hf_token = userdata.get("HF_TOKEN")
except Exception as e:
    raise RuntimeError("Add your Hugging Face token as a Colab secret named HF_TOKEN "
                       "and enable notebook access (key icon in the left sidebar).") from e
login(token=hf_token)

## 4. Load the model

In [ ]:
import torch
from transformers import AutoProcessor, Gemma3ForConditionalGeneration

assert torch.cuda.is_available(), "No GPU found: Runtime → Change runtime type → T4 GPU"
print("GPU:", torch.cuda.get_device_name(0))

# Gemma 3 4B is published as an image+text model; we only send text.
# bfloat16 halves memory (~8–9 GB), so it fits on a free T4.
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = Gemma3ForConditionalGeneration.from_pretrained(
    MODEL_ID, device_map="auto", dtype=torch.bfloat16
).eval()
# Gemma ships sampling defaults; we decode greedily, so clear them to silence the "not valid flags" warning.
model.generation_config.top_p = None
model.generation_config.top_k = None


def generate(messages, max_new_tokens=MAX_NEW_TOKENS):
    """Run one chat turn and return only the newly generated text."""
    inputs = processor.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=True,
        return_dict=True, return_tensors="pt",
    ).to(model.device)
    prompt_len = inputs["input_ids"].shape[-1]
    with torch.inference_mode():
        # do_sample=False = greedy decoding, so the same note gives the same answer every run.
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return processor.decode(out[0][prompt_len:], skip_special_tokens=True)

## 5. Load the notes

In [ ]:
import pandas as pd

if not os.path.exists(NOTES_PATH):
    raise FileNotFoundError("notes.csv not found. Upload data/notes.csv via the Files panel (folder icon).")

notes = pd.read_csv(NOTES_PATH)
required = {"note_id", "sample_name", "transcription", "answer_key_procedures_approx"}
missing = required - set(notes.columns)
assert not missing, f"notes.csv is missing columns: {missing}"
print(f"{len(notes)} notes loaded")
notes[["note_id", "sample_name", "answer_key_procedures_approx"]].head()

## 6. Prompt, JSON parsing and validation

The model must reply with JSON only, and procedure values must be one of the 9 class IDs.

* **Malformed JSON / wrong shape:** ask **once more**, telling the model what was wrong. If the second reply
  is also unusable, the note is recorded as `failed` (raw output kept for debugging) and the run continues.
* **Procedure value not a class ID:** formatting slips like `"Carpal tunnel release"` are normalized to
  `carpal_tunnel_release`. Anything else (e.g. `"osteotomy"`, `"ulnar nerve transposition"`) triggers the
  same one-time retry. If the retry still has invalid values, the valid ones are kept and the invalid
  ones are moved to `rejected_procedures` so the backend can flag them ("value outside allowed options").
  We don't fuzzy-map free text to classes: that would be a second, hidden classifier.
* **Empty evidence quote** on a field whose value isn't `"not mentioned"`: same one-time retry. If the
  quote is still empty, the note is kept and the backend flags that field ("no quote").

Quote checking and the other flag rules are the backend's job; the preview below only gives a quick look.

In [ ]:
import json
import re
import time

# Class IDs must match PROCEDURE_CLASSES in scripts/prepare_data.py.
PROCEDURE_OPTIONS = {
    "arthroplasty": "joint replacement or resurfacing, total or partial (hemiarthroplasty), any joint, "
                    "including resection or interposition arthroplasty",
    "arthroscopy": "any arthroscopic (scope) procedure, e.g. meniscectomy, chondroplasty, arthroscopic labral repair",
    "fracture_fixation": "fixing a fracture: ORIF, closed reduction, pinning, intramedullary nail/rod, external fixation. "
                         "Screw fixation of a planned osteotomy (e.g. in a bunion operation) is NOT fracture_fixation",
    "spinal_fusion": "spinal fusion or arthrodesis of vertebrae, including interbody fusion and TLIF",
    "spinal_decompression": "discectomy, laminectomy, laminotomy, foraminotomy or facetectomy of the spine",
    "carpal_tunnel_release": "cutting the transverse carpal ligament at the wrist (open or endoscopic)",
    "soft_tissue_release": "trigger finger/thumb (A1 pulley) release, de Quervain (first dorsal compartment) "
                           "release, or hamstring release; never carpal tunnel",
    "tendon_ligament_repair": "repair or reconstruction of a tendon or ligament, e.g. rotator cuff repair, "
                              "ACL reconstruction, Achilles tendon repair",
    "bunionectomy_osteotomy": "bunionectomy, or an osteotomy of a foot bone (e.g. Austin, Akin, metatarsal osteotomy)",
}
CLASS_IDS = list(PROCEDURE_OPTIONS)
FIELDS = ["procedure", "body_part", "laterality", "patient_age"]

class_lines = "\n".join(f"  {cid}: {desc}" for cid, desc in PROCEDURE_OPTIONS.items())
id_list = ", ".join(f'"{cid}"' for cid in CLASS_IDS)

SYSTEM_PROMPT = f"""You extract structured data from orthopedic operative notes for a research registry.
Reply with ONLY one JSON object, no markdown fences and no text before or after it.

FIELD 1, procedure: a list of procedure class IDs. Allowed IDs (ID: meaning):
{class_lines}
Rules for procedure:
- Each "value" MUST be one of these exact IDs, copied character for character: {id_list}.
  Never write a procedure name, abbreviation or any other text as the value.
- Include an ID only if the note says that procedure was performed in THIS operation
  (not a diagnosis, a past surgery or a plan). Do not guess from the body part alone.
- Procedures that match no ID (e.g. nerve transposition, bursa excision, hardware removal) are left out.
- A note can have several IDs. If none apply, use an empty list [].

FIELD 2, body_part: the main body part operated on, short and lowercase, without the side
  (e.g. "knee", "hip", "shoulder", "wrist", "hand", "finger", "thumb", "foot", "ankle", "cervical spine", "lumbar spine").
FIELD 3, laterality: exactly one of "left", "right", "bilateral", "not mentioned".
FIELD 4, patient_age: the patient's age in years as an integer, or "not mentioned".

EVIDENCE RULES (apply to every field, including body_part and laterality):
- Every value that is not "not mentioned" MUST have a non-empty "evidence" quote.
- The quote is a short phrase of about 3 to 15 words, copied EXACTLY from the note as one continuous
  piece of text: same words, same order. Do not reword, summarize, fix spelling, join separate
  sentences, or use "...".
- Only when a value is "not mentioned" is its evidence "".

JSON format:
{{
  "procedure": [{{"value": "<one of the allowed IDs>", "evidence": "<exact quote>"}}],
  "body_part": {{"value": "<body part>", "evidence": "<exact quote>"}},
  "laterality": {{"value": "<left|right|bilateral|not mentioned>", "evidence": "<exact quote>"}},
  "patient_age": {{"value": <integer or "not mentioned">, "evidence": "<exact quote>"}}
}}

Worked example. Note:
PROCEDURE: Left knee arthroscopy with partial medial meniscectomy.
HISTORY: The patient is a 54-year-old man with a medial meniscus tear of the left knee.
Correct reply:
{{
  "procedure": [{{"value": "arthroscopy", "evidence": "Left knee arthroscopy with partial medial meniscectomy"}}],
  "body_part": {{"value": "knee", "evidence": "Left knee arthroscopy"}},
  "laterality": {{"value": "left", "evidence": "medial meniscus tear of the left knee"}},
  "patient_age": {{"value": 54, "evidence": "The patient is a 54-year-old man"}}
}}
If that note had no age, patient_age would be {{"value": "not mentioned", "evidence": ""}}."""


def text_msg(role, text):
    # Gemma 3's processor expects content as a list of typed parts.
    return {"role": role, "content": [{"type": "text", "text": text}]}


def build_messages(note_text):
    return [text_msg("system", SYSTEM_PROMPT),
            text_msg("user", f"Operative note:\n\n{note_text}")]


def parse_json(reply):
    """Pull the JSON object out of the reply. Small models often add ```json fences or a preamble."""
    text = re.sub(r"```(?:json)?", "", reply).strip()
    start, end = text.find("{"), text.rfind("}")
    if start == -1 or end <= start:
        raise ValueError("no JSON object found in the reply")
    try:
        return json.loads(text[start:end + 1])
    except json.JSONDecodeError as e:
        raise ValueError(f"invalid JSON: {e}") from e


def check_value_evidence(field, item):
    if not isinstance(item, dict) or "value" not in item or "evidence" not in item:
        raise ValueError(f'{field} must be an object with "value" and "evidence"')
    if not isinstance(item["evidence"], str):
        raise ValueError(f"{field}.evidence must be a string")


def to_class_id(value):
    """Fix formatting only: 'Carpal Tunnel Release' / 'carpal-tunnel release' -> 'carpal_tunnel_release'.
    Returns None if the result still isn't an allowed ID."""
    cid = re.sub(r"[\s\-/]+", "_", value.strip().strip("\"'").lower())
    return cid if cid in PROCEDURE_OPTIONS else None


def validate(obj):
    """Check structure and procedure IDs.

    Raises ValueError for unusable structure. Otherwise returns (fields, rejected), where `rejected`
    holds procedure items whose value isn't an allowed class ID.
    """
    if not isinstance(obj, dict):
        raise ValueError("reply must be a JSON object")
    missing = [f for f in FIELDS if f not in obj]
    if missing:
        raise ValueError(f"missing fields: {missing}")
    if not isinstance(obj["procedure"], list):
        raise ValueError("procedure must be a list")
    for field in ["body_part", "laterality", "patient_age"]:
        check_value_evidence(field, obj[field])
    for field in ["body_part", "laterality"]:
        if not isinstance(obj[field]["value"], str):
            raise ValueError(f"{field}.value must be a string")
    age = obj["patient_age"]["value"]
    if isinstance(age, bool) or not isinstance(age, (int, str)):
        raise ValueError('patient_age.value must be an integer or "not mentioned"')

    procedures, rejected, seen = [], [], set()
    for item in obj["procedure"]:
        check_value_evidence("procedure item", item)
        cid = to_class_id(item["value"]) if isinstance(item["value"], str) else None
        if cid is None:
            rejected.append(item)
        elif cid not in seen:  # drop duplicates of the same class
            seen.add(cid)
            procedures.append({"value": cid, "evidence": item["evidence"]})

    # Light normalization only. Out-of-option laterality etc. is left for the backend to flag.
    if isinstance(age, str) and age.strip().isdigit():
        age = int(age.strip())
    elif isinstance(age, str):
        age = age.strip().lower()
    fields = {
        "procedure": procedures,
        "body_part": {"value": obj["body_part"]["value"].strip().lower(), "evidence": obj["body_part"]["evidence"]},
        "laterality": {"value": obj["laterality"]["value"].strip().lower(), "evidence": obj["laterality"]["evidence"]},
        "patient_age": {"value": age, "evidence": obj["patient_age"]["evidence"]},
    }
    return fields, rejected, missing_quotes(fields)


def missing_quotes(fields):
    """Names of fields that have a real value but an empty evidence quote."""
    missing = [f"procedure {p['value']}" for p in fields["procedure"] if not p["evidence"].strip()]
    for f in ["body_part", "laterality", "patient_age"]:
        v = fields[f]
        if str(v["value"]).strip().lower() != "not mentioned" and not v["evidence"].strip():
            missing.append(f)
    return missing


def extract_note(note_id, note_text):
    """Extract one note: up to 2 attempts. Never raises; failures become a 'failed' record."""
    messages = build_messages(note_text)
    raw_outputs, error, partial = [], None, None
    started = time.time()

    def record(status, fields=None, rejected=()):
        return {"note_id": int(note_id), "status": status, "attempts": len(raw_outputs),
                "fields": fields, "rejected_procedures": list(rejected), "error": error,
                "raw_outputs": raw_outputs, "seconds": round(time.time() - started, 1)}

    for attempt in (1, 2):
        try:
            reply = generate(messages)
        except Exception as e:  # e.g. CUDA out of memory on a very long note
            error = f"generation error: {type(e).__name__}: {e}"
            if "out of memory" in str(e).lower():
                torch.cuda.empty_cache()
            break
        raw_outputs.append(reply)
        try:
            fields, rejected, no_quote = validate(parse_json(reply))
        except ValueError as e:
            error = str(e)
            problem = f"That reply was not valid ({error})."
        else:
            if not rejected and not no_quote:
                error = None
                return record("ok", fields)
            partial = (fields, rejected)  # usable, apart from bad procedure values / missing quotes
            errors, problems = [], []
            if rejected:
                bad = ", ".join(repr(r["value"]) for r in rejected)
                errors.append(f"procedure values not in the allowed list: {bad}")
                problems.append(f"These procedure values are not allowed IDs: {bad}. Each value must be exactly "
                                f"one of: {id_list}. Leave out procedures that match no ID.")
            if no_quote:
                errors.append(f"empty evidence for: {', '.join(no_quote)}")
                problems.append(f"These fields have a value but an empty evidence quote: {', '.join(no_quote)}. "
                                "Every value that is not \"not mentioned\" needs a short exact quote from the note.")
            error = "; ".join(errors)
            problem = " ".join(problems)
        if attempt == 1:
            # Retry once: show the model its own reply and what was wrong with it.
            messages = messages + [
                text_msg("assistant", reply),
                text_msg("user", problem + " Reply again with ONLY the JSON object, nothing else."),
            ]

    if partial:
        # Still imperfect after the retry: keep the usable fields and record the invalid procedure values.
        # Missing quotes stay as "" and the backend flags them.
        return record("ok", *partial)
    return record("failed")

## 7. Run loop with checkpoints

Results are written to disk every `SAVE_EVERY` notes. If Colab disconnects, just re-run this cell and the
run cell: notes that already succeeded are skipped, and failed notes are retried.

In [ ]:
import difflib
from datetime import datetime, timezone


def load_results(path):
    if not os.path.exists(path):
        return []
    with open(path) as f:
        data = json.load(f)
    if data.get("model") != MODEL_ID or data.get("prompt_version") != PROMPT_VERSION:
        raise RuntimeError(f"{path} was made with a different model/prompt. Rename or delete it first.")
    return data["results"]


def save_results(path, results):
    data = {"model": MODEL_ID, "prompt_version": PROMPT_VERSION,
            "updated_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
            "results": sorted(results, key=lambda r: r["note_id"])}
    # Write to a temp file then rename, so a disconnect mid-write can't leave a half-written results.json.
    tmp = path + ".tmp"
    with open(tmp, "w") as f:
        json.dump(data, f, indent=2)
    os.replace(tmp, path)


def run(df, path, fresh=False):
    if fresh and os.path.exists(path):
        os.remove(path)
    # Keep successes from earlier runs; drop earlier failures so they get retried.
    results = [r for r in load_results(path) if r["status"] == "ok"]
    done = {r["note_id"] for r in results}
    todo = df[~df["note_id"].isin(done)]
    print(f"{len(done)} already done, {len(todo)} to process")

    for i, row in enumerate(todo.itertuples(index=False), start=1):
        rec = extract_note(row.note_id, row.transcription)
        results.append(rec)
        note = f"  ({rec['error']})" if rec["error"] else ""
        print(f"[{i}/{len(todo)}] note {rec['note_id']}: {rec['status']} "
              f"in {rec['seconds']}s, attempts={rec['attempts']}{note}")
        if i % SAVE_EVERY == 0:
            save_results(path, results)
            print(f"  checkpoint saved ({len(results)} results)")
    save_results(path, results)
    print(f"Done. {len(results)} results in {path}")
    return results


def normalize_for_match(text):
    """Lowercase and turn every run of punctuation/whitespace into one space.

    So 'Right carpal tunnel release.' matches 'right  carpal-tunnel release', and the model dropping
    a period or changing a comma doesn't count as a misquote. The backend should use the same rule.
    """
    return " ".join(re.sub(r"[^a-z0-9]+", " ", str(text).lower()).split())


NEAR_MATCH = 0.9  # share of quote words that must line up with the note for a near-verbatim match


def quote_match(quote, note_text):
    """How the quote appears in the note: "exact", "near" (near-verbatim), or None (not found)."""
    q, n = normalize_for_match(quote), normalize_for_match(note_text)
    if not q:
        return None
    # Pad with spaces so whole words must match ("right" must not match inside "bright").
    if f" {q} " in f" {n} ":
        return "exact"
    # A quote stitched together with "..." counts if every piece is in the note.
    pieces = [normalize_for_match(p) for p in re.split(r"\.\.\.|…", str(quote))]
    pieces = [p for p in pieces if p]
    if len(pieces) > 1 and all(f" {p} " in f" {n} " for p in pieces):
        return "near"
    # Otherwise slide a window the length of the quote over the note and compare word sequences,
    # so one changed or dropped word in a 10+ word quote still counts. Short quotes must be exact.
    qt, nt = q.split(), n.split()
    if len(qt) >= 5:
        sm = difflib.SequenceMatcher(autojunk=False)
        sm.set_seq2(qt)
        for i in range(max(1, len(nt) - len(qt) + 1)):
            sm.set_seq1(nt[i:i + len(qt)])
            if sm.quick_ratio() >= NEAR_MATCH and sm.ratio() >= NEAR_MATCH:
                return "near"
    return None


# "Not mentioned" is only doubtful if the note seems to contain the information.
AGE_PATTERN = re.compile(r"\b\d{1,3}\s*-?\s*(?:year|yr)s?\s*-?\s*old\b|\bage[d:]?\s+\d{1,3}\b", re.IGNORECASE)
LATERALITY_PATTERN = re.compile(r"\b(?:left|right|bilateral(?:ly)?)\b", re.IGNORECASE)


def field_flag(field, value, quote, note_text):
    """Preview of the backend's per-field flag rules. Returns a short status string."""
    if str(value).strip().lower() == "not mentioned":
        if field == "patient_age":
            hit = AGE_PATTERN.search(note_text)
        elif field == "laterality":
            hit = LATERALITY_PATTERN.search(note_text)
        else:  # every operative note has a body part
            return "flag: not mentioned"
        return f"flag: not mentioned, but note has '{hit.group(0)}'" if hit else "ok (not mentioned)"
    if not str(quote).strip():
        return "flag: no quote"
    match = quote_match(quote, note_text)
    if match == "exact":
        return "quote ok"
    if match == "near":
        return "quote ok (near-verbatim)"
    return "flag: quote not in note"


def preview(results, df):
    by_id = df.set_index("note_id")
    for r in results:
        row = by_id.loc[r["note_id"]]
        note_text = row["transcription"]
        print(f"\n=== note {r['note_id']}: {row['sample_name']} ===")
        print("answer key (approx):", row["answer_key_procedures_approx"])
        if r["status"] != "ok":
            print("FAILED:", r["error"])
            continue
        if not r["fields"]["procedure"]:
            print("  procedure   (none)                   flag: no procedure found")
        for p in r["fields"]["procedure"]:
            print(f"  procedure   {p['value']!s:24} {field_flag('procedure', p['value'], p['evidence'], note_text)}")
        for p in r.get("rejected_procedures", []):
            print(f"  procedure   {p['value']!s:24} flag: not an allowed class ID")
        for f in ["body_part", "laterality", "patient_age"]:
            v = r["fields"][f]
            print(f"  {f:11} {v['value']!s:24} {field_flag(f, v['value'], v['evidence'], note_text)}")


def flag_rate(results, df):
    """Share of extracted fields the preview rules would flag (one field = one procedure or one other field)."""
    by_id = df.set_index("note_id")
    total = flagged = 0
    for r in results:
        if r["status"] != "ok":
            total += 1; flagged += 1  # JSON failure: the whole note goes to review
            continue
        text = by_id.loc[r["note_id"]]["transcription"]
        statuses = [field_flag("procedure", p["value"], p["evidence"], text) for p in r["fields"]["procedure"]]
        statuses += ["flag"] * (len(r.get("rejected_procedures", [])) + (not r["fields"]["procedure"]))
        statuses += [field_flag(f, r["fields"][f]["value"], r["fields"][f]["evidence"], text)
                     for f in ["body_part", "laterality", "patient_age"]]
        total += len(statuses); flagged += sum(s.startswith("flag") for s in statuses)
    print(f"preview: {flagged}/{total} fields flagged ({flagged / max(total, 1):.0%})")

## 8. Test on 5 notes

Run this first. Check that the replies parse, the procedure classes look sensible against the
(approximate) answer key, and that the quotes are found in the notes. The per-note seconds also tell you
roughly how long the full run will take. Writes to `results_test.json`, not the main results file.

In [ ]:
test_results = run(notes.head(5), TEST_RESULTS_PATH, fresh=True)
preview(test_results, notes)
print()
flag_rate(test_results, notes)

## 9. Full run

Saves to `results.json` every 10 notes. Safe to re-run after a disconnect: it resumes where it stopped.

In [ ]:
results = run(notes, RESULTS_PATH)

## 10. Summary and download

In [ ]:
ok = [r for r in results if r["status"] == "ok"]
failed = [r for r in results if r["status"] == "failed"]
print(f"{len(results)} notes: {len(ok)} ok, {len(failed)} failed (JSON failures after one retry)")
print(f"needed a retry: {sum(1 for r in ok if r['attempts'] == 2)}")
print(f"kept with invalid procedure values after retry: {sum(1 for r in ok if r['rejected_procedures'])}")
for r in failed:
    print(f"  note {r['note_id']}: {r['error']}")
flag_rate(results, notes)

from google.colab import files
files.download(RESULTS_PATH)  # then commit it to the repo as data/results.json